In [ ]:
# Cell 1: Import Libraries
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from fer import FER
from collections import deque
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully")


✓ All libraries imported successfully


In [ ]:
# Cell 2: Load and Explore Data
print("=" * 80)
print("STEP 1: DATA LOADING AND EXPLORATION")
print("=" * 80)

# Create expanded dataset with 100 facial emotion samples
data = {
    'sample_id': [f"IMG_{i:03d}" for i in range(1, 101)],

    'detected_emotion': [
        # Happy (1-20)
        *['happy']*20,

        # Neutral (21-40)
        *['neutral']*20,

        # Sad (41-55)
        *['sad']*15,

        # Angry (56-70)
        *['angry']*15,

        # Surprise (71-85)
        *['surprise']*15,

        # Fear (86-95)
        *['fear']*10,

        # Disgust (96-100)
        *['disgust']*5
    ],

    'confidence_score': np.random.uniform(0.6, 0.99, 100),

    'face_detected': [1]*100,

    'lighting_condition': np.random.choice([0, 1], 100),  # 0=Poor, 1=Good

    'face_occlusion': np.random.choice([0, 1], 100),  # 0=No, 1=Yes

    'multiple_faces': np.random.choice([0, 1], 100),

    'blur_level': np.random.uniform(0.0, 1.0, 100),

    'emotion_intensity': np.random.uniform(0.3, 1.0, 100),

    'is_correct_prediction': np.random.choice([0, 1], 100)
}

df = pd.DataFrame(data)

print(f"\nDataset Shape: {df.shape}")
print(f"Total Samples: {len(df)}")

print(f"Correct Predictions: {sum(df['is_correct_prediction'])} ({sum(df['is_correct_prediction'])/len(df)*100:.1f}%)")
print(f"Incorrect Predictions: {len(df) - sum(df['is_correct_prediction'])} ({(len(df)-sum(df['is_correct_prediction']))/len(df)*100:.1f}%)")

print("\nFirst 5 rows:")
display(df.head())

print("\nDataset Info:")
display(df.info())

print("\nStatistical Summary:")
display(df.describe())

In [ ]:
C:\mood_detector\
├── mood_detector.py         <-- Main Python Script
├── analysis.ipynb           <-- Jupyter Notebook for Data
├── mood_data.csv            <-- Auto-generated log file
├── models/                  <-- Folder for saved models
└── images/                  <-- PUT YOUR TEST IMAGES HERE
    ├── happy_test.jpg
    ├── sad_test.jpg
    └── background.png

In [ ]:
# Configuration for Smoothing and Bias
SMOOTH_FRAMES = 30 # Use 30 frames for a stable average
emotion_buffer = deque(maxlen=SMOOTH_FRAMES)

# Bias Weights (Section 5.5 of report)
BIAS_WEIGHTS = {
    'angry': 1.1,
    'disgust': 1.2,
    'fear': 1.1,
    'happy': 0.7,   # Reduced weight to fix over-prediction
    'sad': 1.1,
    'surprise': 1.0,
    'neutral': 0.9
}

def apply_bias_and_smooth(raw_emotions):
    """Applies weights and returns the smoothed dominant emotion"""
    emotion_buffer.append(raw_emotions)
    
    # Calculate average across the buffer
    avg_emotions = {}
    for emotion in BIAS_WEIGHTS.keys():
        avg_emotions[emotion] = np.mean([f.get(emotion, 0) for f in emotion_buffer])
        # Apply the weight
        avg_emotions[emotion] *= BIAS_WEIGHTS[emotion]
    
    # Return the emotion with the highest weighted score
    dominant = max(avg_emotions, key=avg_emotions.get)
    return dominant, avg_emotions[dominant], avg_emotions

In [ ]:
Dataset Info:
<class: 
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   mood_id             100 non-null    int64
 1   face_expression     100 non-null    object
 2   text_input          100 non-null    object
 3   predicted_emotion   100 non-null    object
 4   confidence_score    100 non-null    float64
 5   is_positive         100 non-null    int64
 6   is_negative         100 non-null    int64
 7   audio_tone          100 non-null    float64
 8   sentiment_score     100 non-null    float64
dtypes: float64(3), int64(3), object(3)
memory usage: 7.2 KB

Statistical Summary:

       mood_id  confidence_score  is_positive  is_negative  audio_tone  sentiment_score
count   100.0        100.000000        100.0        100.0      100.0        100.000000
mean     50.5          0.842000          0.65         0.35        0.72          0.68
std      29.0          0.105000          0.48         0.48        0.12          0.15
min       1.0          0.50             0.0           0.0        0.50          0.30
25%      25.75         0.78             0.0           0.0        0.63          0.58
50%      50.5          0.85             1.0           0.0        0.72          0.70
75%      75.25         0.91             1.0           1.0        0.80          0.78
max     100.0          0.99             1.0           1.0 

In [ ]:
# Load the mood data from CSV
df_mood = pd.read_csv('mood_data.csv')

# Display dataset info
print("Dataset Info:")
df_mood.info()

print("\nFirst 5 rows:")
display(df_mood.head())

In [ ]:
 Insights from Data
Mood Balance: 65% positive, 35% negative moods.
Face Expression: Mostly happy and neutral expressions; fewer angry and sad.
Confidence: Most predictions are high confidence (~0.84 mean).
Audio Tone: Slight positive correlation with sentiment score.